# Resource estimation for SOSSA sum-of-squares spectral amplification

This notebook estimates the cost of running **unary-iteration quantum phase estimation** on a
Hamiltonian block-encoded with **SOSSA** (sum-of-squares spectral amplification, Low *et al.*,
[arXiv:2502.15882](https://arxiv.org/abs/2502.15882)).

SOSSA writes an electronic-structure Hamiltonian as a shifted sum of squares,

$$
H \;=\; \underbrace{\sum_{x} c_x\, A_x^{\dagger} A_x}_{H_{\mathrm{gap}} \;\succeq\; 0} \;+\; E_{\mathrm{SOS}},
$$

where every generator $A_x$ is a Majorana bilinear drawn from a double-factorized (DFTHC)
representation of the two-body integrals. Because $H_{\mathrm{gap}}$ is positive semi-definite,
its ground state sits at the *edge* of the walk operator's spectral band, and the phase-to-energy
map

$$
E_{\mathrm{gap}}(\varphi) \;=\; 2\Lambda \cos^{2}(\pi\varphi)
$$

is quadratically flat there. That flatness is the amplification: a phase resolved to
$\delta\varphi \sim 1/T$ yields an energy resolved to $\delta E \sim 1/T^{2}$, so the query count
needed for a target precision scales far better than for a plain qubitization walk.

We run the full pipeline twice:

1. **Part 1 --- a stored DFTHC instance.** A small H<sub>2</sub> Hamiltonian loaded straight from
   JSON, already in the factorized form SOSSA consumes, with a non-zero `energy_gap` so the
   amplified walk is exercised.
2. **Part 2 --- stretched N<sub>2</sub>, built from scratch.** SCF, orbital localization,
   automated active-space selection, then a **double factorization** of the active-space
   two-body integrals to produce the SOSSA input.

Both parts end in the same place: a unary-iteration QPE circuit, its logical gate counts, and a
physical resource estimate from `qdk.qre`.

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md),
you will need the `qre`, `jupyter` and `qiskit-extras` extras:

```bash
pip install 'qdk-chemistry[jupyter,qiskit-extras,qre]'
```

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd

# Reduce logging output for the demo
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef,
    Configuration,
    Hamiltonian,
    MajoranaMapping,
    StateVectorContainer,
    Wavefunction,
)

## The shared pipeline

Both halves of the notebook funnel into the same three steps, so we define them once.

`SOSSAQubitMapper` turns a `FactorizedHamiltonianContainer` into a structured qubit operator that
records the generator one-norms, the block-encoding normalization $\Lambda$, and the scalar shift
$E_{\mathrm{SOS}}$. The unary-iteration QPE builder then consumes that operator directly: it
allocates one phase qubit per query and reuses a single controlled walk across every slot.

The circuit-mapper settings choose how each piece of the walk is synthesized:

- `outer_prepare` --- prepares $\sqrt{c_x}$-weighted amplitudes over the generator index. Alias
  sampling gives a gate count that grows linearly rather than exponentially in the register width.
- `inner_prepare_algorithm` --- the controlled rotation cascade that builds each Majorana bilinear.
- `select_algorithm` --- `qrom_phase_gradient` applies the basis rotations through a QROM lookup
  into a shared phase-gradient register, which trades rotation synthesis for Toffolis.
- `rotation_bit_precision` / `coefficient_bit_precision` --- how finely the Givens angles and the
  amplitudes are discretized. These dominate the T/Toffoli count, so they are the main knobs to
  sweep when tightening a resource estimate.

In [ ]:
def sossa_unary_qpe_circuit(
    hamiltonian,
    *,
    num_queries,
    n_alpha,
    n_beta,
    rotation_bit_precision=15,
    coefficient_bit_precision=11,
):
    """Build a unary-iteration QPE circuit driven by the SOSSA walk.

    Args:
        hamiltonian: A Hamiltonian backed by a FactorizedHamiltonianContainer.
        num_queries: Number of walk applications in the phase-estimation schedule.
        n_alpha: Number of alpha electrons in the reference determinant.
        n_beta: Number of beta electrons in the reference determinant.
        rotation_bit_precision: Bits used to discretize the Givens rotation angles.
        coefficient_bit_precision: Bits used to discretize the PREPARE amplitudes.

    Returns:
        A tuple of (QPE circuit, SOSSA qubit operator).

    """
    container = hamiltonian.get_container()
    num_orbitals = container.get_num_orbitals()
    orbitals = container.get_orbitals()

    operator = create("qubit_mapper", "sossa").run(
        hamiltonian, MajoranaMapping.jordan_wigner(2 * num_orbitals)
    )

    # Hartree-Fock reference determinant, loaded with the sparse isometry method.
    hf_config = Configuration.canonical_hf_configuration(n_alpha, n_beta, num_orbitals)
    reference = Wavefunction(StateVectorContainer(hf_config, orbitals))
    state_prep = create("state_prep", "sparse_isometry").run(reference)

    builder = create(
        "qpe_circuit_builder",
        "qdk_unary",
        num_queries=num_queries,
        circuit_mapper=AlgorithmRef(
            "circuit_mapper",
            "sossa",
            outer_prepare=AlgorithmRef("state_prep", "alias_sampling"),
            inner_prepare_algorithm="controlled_alias_sampling",
            select_algorithm="qrom_phase_gradient",
            rotation_bit_precision=rotation_bit_precision,
            coefficient_bit_precision=coefficient_bit_precision,
        ),
        unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa"),
    )
    circuit = builder.run(state_preparation=state_prep, qubit_hamiltonian=operator)[0]
    return circuit, operator


def heisenberg_queries(lambda_sos, target_precision):
    """Queries needed to resolve ``target_precision`` at the amplified band edge.

    The unary schedule wants ``num_queries + 1`` to be a power of two, so round up.
    """
    ideal = math.pi * lambda_sos / (2.0 * target_precision)
    return 2 ** math.ceil(math.log2(max(ideal, 2.0))) - 1


def describe_factorization(container, label):
    """Summarize the (N, R, B, C) shape and the sum-of-squares precondition."""
    signs = np.asarray(container.get_signs(), dtype=float)
    rows = {
        "spatial orbitals (N)": container.get_num_orbitals(),
        "ranks (R)": container.get_num_ranks(),
        "bases (B)": container.get_num_bases(),
        "copies (C)": container.get_num_copies(),
        "negative signs": int((signs < 0).sum()),
        "energy_gap (Hartree)": container.get_energy_gap(),
        "lambda (Hartree)": container.get_lambda(),
        "core energy (Hartree)": container.get_core_energy(),
    }
    return pd.DataFrame(rows.items(), columns=["Property", label])


def logical_counts_frame(circuit, label):
    """Return the circuit's logical resource counts as a one-column frame."""
    counts = circuit.estimate().logical_counts
    return pd.DataFrame(counts.items(), columns=["Logical Estimate", label]).set_index("Logical Estimate")

> **On the sum-of-squares precondition.** `SOSSAQubitMapper` requires every per-rank sign of the
> factorization to be non-negative --- that is what makes the decomposition a genuine *sum of
> squares* and hence $H_{\mathrm{gap}} \succeq 0$. `describe_factorization` reports the negative-sign
> count explicitly rather than letting a violation surface later as an opaque error. For a
> factorization of a true electron-repulsion integral tensor this count should be zero, because
> the ERI supermatrix is a Coulomb Gram matrix and therefore positive semi-definite.

## Part 1 --- a stored DFTHC Hamiltonian

The first instance ships with the repository as a serialized `FactorizedHamiltonianContainer`.
It is a minimal H<sub>2</sub> problem, $N = 2$ spatial orbitals with $R = 1$ rank, $B = 2$ bases
and $C = 1$ copy, and it carries a non-zero `energy_gap`, which is the spectral-amplification
parameter: the walk is built to amplify around that gap rather than around the raw band.

In [ ]:
json_path = Path("data") / "h2_factorized_r1_b2_c1.hamiltonian.json"
h2_hamiltonian = Hamiltonian.from_json(json_path.read_text())
h2_container = h2_hamiltonian.get_container()

display(describe_factorization(h2_container, "H2 (from JSON)"))

### Mapping to the SOSSA qubit operator

The mapper reads the factorization and emits the generator list together with the metadata the
walk needs: $\Lambda$ (half the sum of squared generator one-norms, which sets the block-encoding
normalization) and $E_{\mathrm{SOS}}$ (the scalar that converts a $H_{\mathrm{gap}}$ eigenvalue back
into a physical energy).

In [ ]:
h2_operator = create("qubit_mapper", "sossa").run(
    h2_hamiltonian, MajoranaMapping.jordan_wigner(2 * h2_container.get_num_orbitals())
)
h2_metadata = h2_operator.get_container().metadata

print(f"Block-encoding normalization  Lambda = {h2_metadata.normalization:.6f} Hartree")
print(f"Sum-of-squares shift         E_SOS  = {h2_metadata.energy_shift:.6f} Hartree")
print(f"System qubits                       = {h2_operator.get_container().num_qubits}")

### Choosing the query schedule

Unary-iteration QPE applies the walk `num_queries` times and reads the phase off a register of
the same width. To resolve an energy $\sigma_E$ at the amplified band edge we need roughly
$\pi \Lambda / (2\sigma_E)$ queries; we round up to the next power of two minus one so the
schedule divides evenly.

In [ ]:
TARGET_PRECISION = 1e-3  # Hartree ("chemical accuracy" is ~1.6e-3)

h2_queries = heisenberg_queries(h2_metadata.normalization, TARGET_PRECISION)
print(f"Queries for {TARGET_PRECISION:.0e} Ha at Lambda = {h2_metadata.normalization:.4f}: {h2_queries}")

h2_circuit, _ = sossa_unary_qpe_circuit(
    h2_hamiltonian,
    num_queries=h2_queries,
    n_alpha=1,
    n_beta=1,
)
h2_counts = logical_counts_frame(h2_circuit, "H2 SOSSA unary QPE")
display(h2_counts)

### Physical resource estimates

`qdk.qre` maps the logical circuit onto a fault-tolerant architecture and returns the
Pareto-optimal trade-offs between physical qubit count and runtime. We use a Majorana-based
architecture at a $10^{-5}$ physical error rate with the `ThreeAux` code and round-based magic
state factories, and budget 1% total error.

In [ ]:
from qdk.qre import estimate, plot_estimates
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

architecture = Majorana(error_rate=1e-5)
isa_query = ThreeAux.q() * RoundBasedFactory.q(use_cache=True, code_query=ThreeAux.q())

h2_results = estimate(
    h2_circuit.get_qre_application(), architecture, isa_query, max_error=0.01, name="SOSSA_H2"
)
h2_results.add_factory_summary_column()
display(h2_results.as_frame())

plot_estimates(h2_results, figsize=(6, 4), runtime_unit="ms")

## Part 2 --- stretched N<sub>2</sub> from a structure file

The second instance is built end to end. Stretching the N<sub>2</sub> bond introduces strong
multi-reference character, which is exactly the regime where a classical single-reference method
struggles and phase estimation is interesting.

The route to a SOSSA-ready Hamiltonian is:
SCF $\rightarrow$ valence space $\rightarrow$ MP2 natural-orbital localization $\rightarrow$
autoCAS-EOS active-space selection $\rightarrow$ active-space Hamiltonian $\rightarrow$
**double factorization**.

In [ ]:
from qdk_chemistry.data import Structure
from qdk_chemistry.data.symmetry import SymmetryLabel, axes

# Stretched N2 structure at 1.270025 Angstrom bond length
structure = Structure.from_xyz_file(Path("data/stretched_n2.structure.xyz"))

scf_solver = create("scf_solver")
E_hf, wfn_hf = scf_solver.run(
    structure,
    charge=0,
    spin_multiplicity=1,
    basis_or_guess="cc-pvdz",
)
print(f"Hartree-Fock energy: {E_hf:.6f} Hartree")

In [ ]:
from qdk_chemistry.utils import compute_valence_space_parameters

# Restrict to the valence space, then localize with MP2 natural orbitals
num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
active_space_selector = create(
    "active_space_selector",
    "qdk_valence",
    num_active_electrons=num_val_e,
    num_active_orbitals=num_val_o,
)
valence_wf = active_space_selector.run(wfn_hf)

localizer = create("orbital_localizer", "qdk_mp2_natural_orbitals")
valence_indices = valence_wf.get_orbitals().active_indices()
loc_wfn = localizer.run(
    valence_wf,
    list(valence_indices.indices(SymmetryLabel([axes.alpha()]))),
    list(valence_indices.indices(SymmetryLabel([axes.beta()]))),
)
print(f"Valence space: {num_val_e} electrons in {num_val_o} orbitals")

In [ ]:
# Selected-CI wavefunction on the localized orbitals, used to drive active-space selection
hamiltonian_constructor = create("hamiltonian_constructor")
loc_hamiltonian = hamiltonian_constructor.run(loc_wfn.get_orbitals())
num_alpha_electrons, num_beta_electrons = loc_wfn.get_active_num_electrons()

macis_mc = create(
    "multi_configuration_calculator",
    "macis_asci",
    calculate_one_rdm=True,
    calculate_two_rdm=True,
)
_, wfn_sci = macis_mc.run(loc_hamiltonian, num_alpha_electrons, num_beta_electrons)

# Entropy-based active space selection
autocas = create("active_space_selector", "qdk_autocas_eos")
autocas_wfn = autocas.run(wfn_sci)
indices = list(autocas_wfn.get_orbitals().active_indices().indices(SymmetryLabel([axes.alpha()])))
print(f"autoCAS-EOS selected {len(indices)} of {num_val_o} orbitals: indices={indices}")

In [ ]:
# Active-space Hamiltonian, plus a CASCI reference energy to benchmark against
refined_orbitals = autocas_wfn.get_orbitals()
active_hamiltonian = hamiltonian_constructor.run(refined_orbitals)

alpha_electrons, beta_electrons = autocas_wfn.get_active_num_electrons()
mc = create("multi_configuration_calculator", "macis_cas")
e_cas, wfn_cas = mc.run(active_hamiltonian, alpha_electrons, beta_electrons)
print(f"Active space CASCI energy: {e_cas:.6f} Hartree")
print(f"Active space: {alpha_electrons} alpha + {beta_electrons} beta electrons")

### Double factorization

SOSSA consumes a `FactorizedHamiltonianContainer`, so the active-space two-body integrals have to
be decomposed first. The eigen-decomposition factorizer diagonalizes the ERI supermatrix and keeps
the ranks above `truncation_threshold`.

We set the threshold to $10^{-8}$ rather than leaving it at the default $10^{-12}$ for two
reasons. It keeps $R$ --- and therefore the walk's PREPARE register and gate count --- small at a
cost far below chemical accuracy; and it discards the round-off-scale eigenvalues whose sign is
numerically meaningless, which is what keeps the sum-of-squares precondition clean.

In [ ]:
factorizer = create("double_factorizer", "eigen_decomposition")
factorizer.settings().set("truncation_threshold", 1e-8)
n2_hamiltonian = factorizer.run(active_hamiltonian)
n2_container = n2_hamiltonian.get_container()

display(describe_factorization(n2_container, "N2 (double factorized)"))

Note the contrast with Part 1: the double factorizer emits `energy_gap = 0`, so this walk is *not*
spectrally amplified around a known gap. Supplying a gap estimate --- for instance from the CASCI
solve above, or from a cheaper correlated method --- is what unlocks the amplified query scaling
on a freshly factorized Hamiltonian.

In [ ]:
n2_operator = create("qubit_mapper", "sossa").run(
    n2_hamiltonian, MajoranaMapping.jordan_wigner(2 * n2_container.get_num_orbitals())
)
n2_metadata = n2_operator.get_container().metadata

print(f"Block-encoding normalization  Lambda = {n2_metadata.normalization:.6f} Hartree")
print(f"Sum-of-squares shift         E_SOS  = {n2_metadata.energy_shift:.6f} Hartree")
print(f"System qubits                       = {n2_operator.get_container().num_qubits}")

### Building the N<sub>2</sub> circuit

$\Lambda$ is much larger here than for H<sub>2</sub>, so a full Heisenberg-limited schedule would
need a correspondingly larger query count. `DEMO_QUERY_CAP` keeps the notebook interactive; raise
or remove it to estimate the cost of a production-precision run.

In [ ]:
DEMO_QUERY_CAP = 255

n2_ideal_queries = heisenberg_queries(n2_metadata.normalization, TARGET_PRECISION)
n2_queries = min(n2_ideal_queries, DEMO_QUERY_CAP)
print(f"Heisenberg-limited queries for {TARGET_PRECISION:.0e} Ha: {n2_ideal_queries}")
print(f"Using {n2_queries} queries for this demo")

n2_circuit, _ = sossa_unary_qpe_circuit(
    n2_hamiltonian,
    num_queries=n2_queries,
    n_alpha=alpha_electrons,
    n_beta=beta_electrons,
)
n2_counts = logical_counts_frame(n2_circuit, "N2 SOSSA unary QPE")
display(n2_counts)

In [ ]:
n2_results = estimate(
    n2_circuit.get_qre_application(), architecture, isa_query, max_error=0.01, name="SOSSA_N2"
)
n2_results.add_factory_summary_column()
display(n2_results.as_frame())

plot_estimates(n2_results, figsize=(6, 4), runtime_unit="ms")

## Comparing the two instances

The logical counts put the two problems side by side. The dominant scaling levers are the
factorization shape $(N, R, B, C)$, which sets how much data the walk's PREPARE and SELECT must
load, and the bit precisions, which set how expensive each rotation is.

In [ ]:
comparison = pd.concat([h2_counts, n2_counts], axis=1)
display(comparison)

summary = pd.DataFrame(
    {
        "H2 (from JSON)": {
            "spatial orbitals (N)": h2_container.get_num_orbitals(),
            "ranks (R)": h2_container.get_num_ranks(),
            "lambda (Hartree)": round(h2_metadata.normalization, 6),
            "energy_gap (Hartree)": h2_container.get_energy_gap(),
            "queries": h2_queries,
        },
        "N2 (double factorized)": {
            "spatial orbitals (N)": n2_container.get_num_orbitals(),
            "ranks (R)": n2_container.get_num_ranks(),
            "lambda (Hartree)": round(n2_metadata.normalization, 6),
            "energy_gap (Hartree)": n2_container.get_energy_gap(),
            "queries": n2_queries,
        },
    }
)
display(summary)

## Where to go next

- **Sweep the bit precisions.** `rotation_bit_precision` and `coefficient_bit_precision` trade
  synthesis cost against the error they inject into the walk. They are usually the cheapest way to
  move a resource estimate.
- **Supply an `energy_gap`.** The double-factorized Hamiltonian above is unamplified. Feeding a
  gap estimate into the container is what turns on the $1/T^{2}$ energy scaling that motivates
  SOSSA in the first place.
- **Tighten the truncation.** Lowering `truncation_threshold` keeps more ranks and a more faithful
  Hamiltonian, at a directly observable cost in $R$ and in gate count.
- **Compare against qubitization.** The `lcu` unitary builder with the `prepare_select_prepare`
  circuit mapper block-encodes the same Hamiltonian without the sum-of-squares structure, which
  makes for a like-for-like baseline under the same architecture and error budget.